# Phase 1: Data Lake Optimization & Athena Cataloging
**AAI-540 ML Ops - Group 8**  
**Author:** Jagadeesh Kumar Sellappan  

## Overview
This notebook executes the foundational data engineering pipeline for the Yelp Sentiment Analysis project. It transitions the architecture from a raw "Bronze" storage layer to an optimized "Silver" analytical data lake. 

## Architectural Objectives
* **Cloud-Native Streaming:** Reads massive raw JSON files (up to 5.3 GB) directly from Amazon S3 in manageable chunks, completely bypassing local SageMaker EBS storage limits to prevent Out-Of-Memory (OOM) kernel crashes.
* **Flattening & Optimization:** Flattens complex, nested JSON structures within the `business` and `user` datasets and converts all files into a compressed, columnar Parquet format.
* **Serverless Cataloging:** Utilizes AWS Data Wrangler (`awswrangler`) to programmatically generate an AWS Glue Data Catalog, mapping the optimized Parquet directories into queryable SQL tables within Amazon Athena.

By fully decoupling the heavy data processing from the local compute environment, this phase ensures that downstream Exploratory Data Analysis (EDA) and Feature Engineering pipelines are highly performant, scalable, and cost-effective.

## Data Sources — Yelp Open Dataset (5 Files)

This notebook ingests all five files from the Yelp Open Dataset, each streamed and chunked directly from S3 to avoid OOM kernel crashes on the local SageMaker EBS volume.

| File | Size | Records | Role in Pipeline |
|---|---|---|---|
| `review.json` | ~5.3 GB | 6.9M | **Primary** — review text, star ratings (sentiment label source), `user_id`, `business_id` |
| `business.json` | ~118 MB | 150K | **Dimension** — business category, city, state, average star rating |
| `user.json` | ~3.4 GB | 1.9M | **Dimension** — reviewer profile: `review_count`, `average_stars`, `elite` status |
| `tip.json` | ~180 MB | 908K | **Supplementary** — short-form tip text, not currently used in modeling |
| `checkin.json` | ~287 MB | 131K | **Supplementary** — venue check-in timestamps, not currently used in modeling |

**Output:** Each file is converted to compressed, columnar Parquet and written to `s3://{BUCKET}/processed-parquet/{table_name}/`, then registered as an external table in the `yelp_sentiment_db` Glue/Athena database via `CREATE EXTERNAL TABLE IF NOT EXISTS` — safe to re-run without creating duplicate table definitions.

# Environment Setup

In [1]:
!pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops sagemaker-studio

Found existing installation: sagemaker-core 2.12.0
Uninstalling sagemaker-core-2.12.0:


  Successfully uninstalled sagemaker-core-2.12.0


Found existing installation: sagemaker-train 1.11.0
Uninstalling sagemaker-train-1.11.0:
  Successfully uninstalled sagemaker-train-1.11.0


Found existing installation: sagemaker-serve 1.11.0
Uninstalling sagemaker-serve-1.11.0:
  Successfully uninstalled sagemaker-serve-1.11.0


Found existing installation: sagemaker-mlops 1.11.0
Uninstalling sagemaker-mlops-1.11.0:
  Successfully uninstalled sagemaker-mlops-1.11.0
Found existing installation: sagemaker_studio 1.1.19


Uninstalling sagemaker_studio-1.1.19:
  Successfully uninstalled sagemaker_studio-1.1.19


In [2]:
!pip install "sagemaker-core==1.0.78" --quiet

In [3]:
!pip install "sagemaker==2.232.2" --no-deps --quiet

In [4]:
!pip install awswrangler --quiet

In [6]:
!pip install boto3 botocore s3transfer pathos schema importlib-metadata --quiet

In [1]:
import pandas as pd
import boto3
import json
import os
import awswrangler as wr

In [2]:
# 1. Configuration
# Unified project bucket (us-east-1) - standardized across all notebooks
bucket = 'aai-540-group8-yelp-data-301798465569-us-east-1-an'
raw_prefix = 'raw-data'
processed_prefix = 'processed-parquet'

# Region standardized to us-east-1 to match the unified project bucket.
# setup_default_session ensures awswrangler (wr.catalog / wr.athena) also uses us-east-1.
boto_session = boto3.Session(region_name='us-east-1')
boto3.setup_default_session(region_name='us-east-1')

s3_client = boto_session.client('s3')
print("Starting Cloud-Native Parquet Conversion Pipeline...\n")

Starting Cloud-Native Parquet Conversion Pipeline...



# Flatten, Partition, and Convert

In [3]:
def process_business_from_s3():
    print("Streaming Business data from S3...")
    business_records = []
    
    # Fetch the object from S3
    response = s3_client.get_object(Bucket=bucket, Key=f"{raw_prefix}/business/business.json")
    
    # Read the streaming body line by line
    for line in response['Body'].iter_lines():
        if line:
            business_records.append(json.loads(line.decode('utf-8')))

    df_bus = pd.json_normalize(business_records)
    df_bus.columns = [c.replace('.', '_').replace(' ', '_').lower() for c in df_bus.columns]

    # Convert to Parquet locally, then upload
    local_parquet = 'business.parquet'
    df_bus.to_parquet(local_parquet, index=False)
    
    s3_destination = f"{processed_prefix}/business/business.parquet"
    s3_client.upload_file(local_parquet, bucket, s3_destination)
    import os
    os.remove(local_parquet)
    print(f"Business Parquet uploaded to s3://{bucket}/{s3_destination}\n")

In [6]:
def process_reviews_from_s3():
    print("Streaming and Chunking Review data from S3...")
    chunk_size = 500000 
    
    import awswrangler as wr
    
    s3_uri = f"s3://{bucket}/{raw_prefix}/review/review.json"
    
    # Read JSON in chunks directly from S3
    chunk_iter = wr.s3.read_json(path=s3_uri, lines=True, chunksize=chunk_size)

    for i, chunk in enumerate(chunk_iter):
        filename = f"review_part_{i}.parquet"
        
        chunk.columns = [c.replace('.', '_').replace(' ', '_').lower() for c in chunk.columns]
        
        chunk.to_parquet(filename, index=False)
        
        s3_destination = f"{processed_prefix}/review/{filename}"
        s3_client.upload_file(filename, bucket, s3_destination)
        
        import os
        os.remove(filename)

        print(f"Review Parquet uploaded to s3://{bucket}/{s3_destination}\n")

In [5]:
# 2. Execute
process_business_from_s3()

Streaming Business data from S3...


Business Parquet uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/processed-parquet/business/business.parquet



In [7]:
process_reviews_from_s3()

Streaming and Chunking Review data from S3...


2026-06-21 07:21:41,038	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 1938796544 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=4.56gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-21 07:21:42,186	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Review Parquet uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/processed-parquet/review/review_part_0.parquet



...

Review Parquet uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/processed-parquet/review/review_part_9.parquet



Review Parquet uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/processed-parquet/review/review_part_10.parquet



Review Parquet uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/processed-parquet/review/review_part_11.parquet



Review Parquet uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/processed-parquet/review/review_part_12.parquet



Review Parquet uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/processed-parquet/review/review_part_13.parquet



In [8]:
# Process the Massive User File  
def process_users_from_s3():
    print("Streaming and Chunking User data from S3 (This will take a few minutes)...")
    
    # We use a smaller chunk size  
    chunk_size = 250000 
    s3_uri = f"s3://{bucket}/{raw_prefix}/user/user.json"
    
    # Read JSON in chunks directly from S3
    chunk_iter = wr.s3.read_json(path=s3_uri, lines=True, chunksize=chunk_size)

    for i, chunk in enumerate(chunk_iter):
        filename = f"user_part_{i}.parquet"
        
        # Sanitize column names for Athena
        chunk.columns = [c.replace('.', '_').replace(' ', '_').lower() for c in chunk.columns]
        
        # Save locally as Parquet
        chunk.to_parquet(filename, index=False)
        
        # Push to S3
        s3_destination = f"{processed_prefix}/user/{filename}"
        s3_client.upload_file(filename, bucket, s3_destination)
        
        # Clean up local disk
        os.remove(filename)
        print(f"Uploaded {filename} to S3...")
    print("User data conversion complete!\n")


# Process the Lighter Metadata Files
def process_light_file_from_s3(file_name):
    print(f"Streaming {file_name.capitalize()} data from S3...")
    s3_uri = f"s3://{bucket}/{raw_prefix}/{file_name}/{file_name}.json"
    
    # Read the entire JSON directly into memory
    df = wr.s3.read_json(path=s3_uri, lines=True)

    # Sanitize column names for Athena
    df.columns = [c.replace('.', '_').replace(' ', '_').lower() for c in df.columns]

    # Convert and upload
    local_parquet = f"{file_name}.parquet"
    df.to_parquet(local_parquet, index=False)
    
    s3_destination = f"{processed_prefix}/{file_name}/{local_parquet}"
    s3_client.upload_file(local_parquet, bucket, s3_destination)
    
    # Clean up local disk
    os.remove(local_parquet)
    print(f"{file_name.capitalize()} Parquet uploaded to s3://{bucket}/{s3_destination}\n")

In [9]:
# 4. Execute the Pipeline
try:
    process_users_from_s3()
    process_light_file_from_s3('tip')
    process_light_file_from_s3('checkin')
    print("All remaining files successfully converted to Parquet!")
except Exception as e:
    print(f"Pipeline failed: {e}")

Streaming and Chunking User data from S3 (This will take a few minutes)...


Uploaded user_part_0.parquet to S3...


Uploaded user_part_1.parquet to S3...


Uploaded user_part_2.parquet to S3...


Uploaded user_part_3.parquet to S3...


...

Uploaded user_part_5.parquet to S3...


Uploaded user_part_6.parquet to S3...


Uploaded user_part_7.parquet to S3...
User data conversion complete!

Streaming Tip data from S3...


Tip Parquet uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/processed-parquet/tip/tip.parquet

Streaming Checkin data from S3...


Checkin Parquet uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/processed-parquet/checkin/checkin.parquet

All remaining files successfully converted to Parquet!


# Create the Parquet-Optimized Athena Tables

In [10]:
# Configuration
DATABASE = 'yelp_sentiment_db'
BUCKET = 'aai-540-group8-yelp-data-301798465569-us-east-1-an'

print(f"Creating Athena Database: {DATABASE}...")
wr.catalog.create_database(name=DATABASE, exist_ok=True)

Creating Athena Database: yelp_sentiment_db...


In [11]:
# Because we flattened the data and used Parquet, the schema is much simpler and faster.
# We map directly to the folders containing the parquet files.
queries = {
    "reviews_parquet": f"""
        CREATE EXTERNAL TABLE IF NOT EXISTS {DATABASE}.reviews_parquet (
            review_id string,
            user_id string,
            business_id string,
            stars bigint,
            useful bigint,
            funny bigint,
            cool bigint,
            text string,
            date timestamp
        )
        STORED AS PARQUET
        LOCATION 's3://{BUCKET}/processed-parquet/review/'
    """,
    
    "businesses_parquet": f"""
        CREATE EXTERNAL TABLE IF NOT EXISTS {DATABASE}.businesses_parquet (
            business_id string,
            name string,
            address string,
            city string,
            state string,
            postal_code string,
            latitude double,
            longitude double,
            stars double,
            review_count bigint,
            is_open bigint,
            categories string  
        )
        STORED AS PARQUET
        LOCATION 's3://{BUCKET}/processed-parquet/business/'
    """,
    "users_parquet": f"""
        CREATE EXTERNAL TABLE IF NOT EXISTS {DATABASE}.users_parquet (
            user_id string,
            name string,
            review_count bigint,
            yelping_since string,
            useful bigint,
            funny bigint,
            cool bigint,
            elite string,
            friends string,
            fans bigint,
            average_stars double
        )
        STORED AS PARQUET
        LOCATION 's3://{BUCKET}/processed-parquet/user/'
    """,
    
    "tips_parquet": f"""
        CREATE EXTERNAL TABLE IF NOT EXISTS {DATABASE}.tips_parquet (
            user_id string,
            business_id string,
            text string,
            date string,
            compliment_count bigint
        )
        STORED AS PARQUET
        LOCATION 's3://{BUCKET}/processed-parquet/tip/'
    """,
    
    "checkins_parquet": f"""
        CREATE EXTERNAL TABLE IF NOT EXISTS {DATABASE}.checkins_parquet (
            business_id string,
            date string
        )
        STORED AS PARQUET
        LOCATION 's3://{BUCKET}/processed-parquet/checkin/'
    """
}

In [12]:
print("Creating remaining Parquet tables in Athena...")
for table_name, query in queries.items():
    wr.athena.start_query_execution(sql=query, database=DATABASE, wait=True)
    print(f"[{table_name}] mapped successfully.")

Creating remaining Parquet tables in Athena...


[reviews_parquet] mapped successfully.


[businesses_parquet] mapped successfully.


[users_parquet] mapped successfully.


[tips_parquet] mapped successfully.


[checkins_parquet] mapped successfully.


# Sanity Check

In [13]:
import awswrangler as wr

# Define your environment parameters
DATABASE_NAME = 'yelp_sentiment_db'
OUTPUT_S3_PATH = 's3://aai-540-group8-yelp-data-301798465569-us-east-1-an/athena-results/'

# Define your test query
test_query = """
SELECT name, city, stars, review_count 
FROM "businesses_parquet" 
LIMIT 10;
"""

# Run the query and return a Pandas DataFrame
df = wr.athena.read_sql_query(
    sql=test_query, 
    database=DATABASE_NAME,
    ctas_approach=False, # Keeps the execution clean for simple SELECTs
    s3_output=OUTPUT_S3_PATH
)

# Display the data
print(f"Successfully retrieved {len(df)} rows!")
print(df.head())

Successfully retrieved 10 rows!
                       name           city  stars  review_count
0  Abby Rappoport, LAC, CMQ  Santa Barbara    5.0             7
1             The UPS Store         Affton    3.0            15
2                    Target         Tucson    3.5            22
3        St Honore Pastries   Philadelphia    4.0            80
4  Perkiomen Valley Brewery     Green Lane    4.5            13


In [14]:
tables = [
    'reviews_parquet',
    'businesses_parquet',
    'users_parquet',
    'tips_parquet',
    'checkins_parquet'
]

print(f"{'Table':<22} {'Row Count':>12}")
print("-" * 36)

for table in tables:
    try:
        count_query = f'SELECT COUNT(*) AS row_count FROM "{table}"'
        df_count = wr.athena.read_sql_query(
            sql=count_query,
            database=DATABASE_NAME,
            ctas_approach=False,
            s3_output=OUTPUT_S3_PATH
        )
        count = df_count['row_count'].iloc[0]
        print(f"{table:<22} {count:>12,}")
    except Exception as e:
        print(f"{table:<22} {'ERROR':>12}  — {e}")

Table                     Row Count
------------------------------------


reviews_parquet           6,990,280


businesses_parquet          150,346


users_parquet             1,987,897


tips_parquet                908,915


checkins_parquet            131,930
